In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/TPC-A/00_setup_t1

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
entity = "region"
key_col = "r_regionkey"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Primary key validation rules: key must not be null and must not be duplicated
key_count = Window.partitionBy(key_col)

validated_df = (
    raw_df
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "region key is null")
        .when(F.col("_key_count") > 1, "region key is duplicated")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

# print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
# print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "nation"
key_col = "n_nationkey"
fk_col = "n_regionkey"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"
parent_silver_tbl = f"{catalog}.silver.region"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Region keys available in the silver region table
parent_keys = spark.table(parent_silver_tbl).select(F.col("r_regionkey").alias("_parent_regionkey"))

# Validation rules: nation key not null, not duplicated, and its region key must exist in silver region
key_count = Window.partitionBy(key_col)

validated_df = (
    raw_df
    .join(parent_keys, F.col(fk_col) == F.col("_parent_regionkey"), "left")
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "nation key is null")
        .when(F.col("_key_count") > 1, "nation key is duplicated")
        .when(F.col("_parent_regionkey").isNull(), "region key not found in silver region")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "validation_status", "_parent_regionkey").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count", "_parent_regionkey").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

# print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
# print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "airline"
key_col = "al_airlineid"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Primary key validation rules: key must not be null and must not be duplicated
key_count = Window.partitionBy(key_col)

validated_df = (
    raw_df
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "airline key is null")
        .when(F.col("_key_count") > 1, "airline key is duplicated")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

# print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
# print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "route"
key_col = "ro_routeid"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Primary key validation rules: key must not be null; if duplicated, keep only the latest record
key_count = Window.partitionBy(key_col)
latest_rank = Window.partitionBy(key_col).orderBy(F.col("_ingested_at").desc(), F.col("_row_id").desc())

raw_df = raw_df.withColumn("_row_id", F.monotonically_increasing_id())

validated_df = (
    raw_df
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn("_recency_rank", F.row_number().over(latest_rank))
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "route key is null")
        .when((F.col("_key_count") > 1) & (F.col("_recency_rank") > 1), "route key is duplicated - older record, latest kept")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "_recency_rank", "_row_id", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count", "_recency_rank", "_row_id").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

# print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
# print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "airline_route"
key_cols = ["ar_airlineid", "ar_routeid"]

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Reference keys available in the silver layer
airline_keys = spark.table(f"{catalog}.silver.airline").select(F.col("al_airlineid").alias("_fk_airlineid"))
route_keys = spark.table(f"{catalog}.silver.route").select(F.col("ro_routeid").alias("_fk_routeid"))

# Validation rules:
# 1. composite key (airline id + route id) must not contain nulls
# 2. if composite key is duplicated, keep only the latest record
# 3. airline id and route id must exist in the silver airline / route tables
key_count = Window.partitionBy(key_cols)
latest_rank = Window.partitionBy(key_cols).orderBy(F.col("_ingested_at").desc(), F.col("_row_id").desc())

raw_df = raw_df.withColumn("_row_id", F.monotonically_increasing_id())

validated_df = (
    raw_df
    .join(airline_keys, F.col("ar_airlineid") == F.col("_fk_airlineid"), "left")
    .join(route_keys, F.col("ar_routeid") == F.col("_fk_routeid"), "left")
    .withColumn("_key_count", F.count(F.lit(1)).over(key_count))
    .withColumn("_recency_rank", F.row_number().over(latest_rank))
    .withColumn(
        "validation_status",
        F.when(F.col("ar_airlineid").isNull() | F.col("ar_routeid").isNull(), "airline id or route id is null")
        .when((F.col("_key_count") > 1) & (F.col("_recency_rank") > 1), "composite key duplicated - older record, latest kept")
        .when(F.col("_fk_airlineid").isNull(), "airline id not found in silver airline")
        .when(F.col("_fk_routeid").isNull(), "route id not found in silver route")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "_recency_rank", "_row_id", "_fk_airlineid", "_fk_routeid", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count", "_recency_rank", "_row_id", "_fk_airlineid", "_fk_routeid").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "passenger"
key_col = "pa_passengerid"
fk_col = "pa_nationkey"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"
parent_silver_tbl = f"{catalog}.silver.nation"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Nation keys available in the silver nation table
parent_keys = spark.table(parent_silver_tbl).select(F.col("n_nationkey").alias("_parent_nationkey"))

# Validation rules:
# 1. passenger id must not be null; if duplicated, keep only the latest record
# 2. nation id must exist in the silver nation table
key_count = Window.partitionBy(key_col)
latest_rank = Window.partitionBy(key_col).orderBy(F.col("_ingested_at").desc(), F.col("_row_id").desc())

raw_df = raw_df.withColumn("_row_id", F.monotonically_increasing_id())

validated_df = (
    raw_df
    .join(parent_keys, F.col(fk_col) == F.col("_parent_nationkey"), "left")
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn("_recency_rank", F.row_number().over(latest_rank))
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "passenger id is null")
        .when((F.col("_key_count") > 1) & (F.col("_recency_rank") > 1), "passenger id is duplicated - older record, latest kept")
        .when(F.col("_parent_nationkey").isNull(), "nation id not found in silver nation")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "_recency_rank", "_row_id", "_parent_nationkey", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count", "_recency_rank", "_row_id", "_parent_nationkey").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "booking"
key_col = "bk_bookingid"
fk_col = "bk_passengerid"

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"
parent_silver_tbl = f"{catalog}.silver.passenger"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Passenger keys available in the silver passenger table
parent_keys = spark.table(parent_silver_tbl).select(F.col("pa_passengerid").alias("_parent_passengerid"))

# Validation rules:
# 1. booking id must not be null; if duplicated, keep only the latest record
# 2. passenger id must exist in the silver passenger table
# 3. flight date must be between 1994-01-01 and 1998-12-31
# 4. flight date must be on or after the booking date
# 5. total price must be greater than 0
key_count = Window.partitionBy(key_col)
latest_rank = Window.partitionBy(key_col).orderBy(F.col("_ingested_at").desc(), F.col("_row_id").desc())

raw_df = raw_df.withColumn("_row_id", F.monotonically_increasing_id())

# Parsed fields used by the validation rules
booking_date = F.to_date("bk_bookingdate", "yyyy-MM-dd")
flight_date = F.to_date("bk_flightdate", "yyyy-MM-dd")
total_price = F.col("bk_totalprice").cast("double")

validated_df = (
    raw_df
    .join(parent_keys, F.col(fk_col) == F.col("_parent_passengerid"), "left")
    .withColumn("_key_count", F.count(key_col).over(key_count))
    .withColumn("_recency_rank", F.row_number().over(latest_rank))
    .withColumn("_booking_date", booking_date)
    .withColumn("_flight_date", flight_date)
    .withColumn("_total_price", total_price)
    .withColumn(
        "validation_status",
        F.when(F.col(key_col).isNull(), "booking id is null")
        .when((F.col("_key_count") > 1) & (F.col("_recency_rank") > 1), "booking id is duplicated - older record, latest kept")
        .when(F.col("_parent_passengerid").isNull(), "passenger id not found in silver passenger")
        .when(F.col("_booking_date").isNull(), "booking date is invalid or null")
        .when(F.col("_flight_date").isNull(), "flight date is invalid or null")
        .when((F.col("_flight_date") < F.lit("1994-01-01")) | (F.col("_flight_date") > F.lit("1998-12-31")), "flight date is outside range 1994-01-01 to 1998-12-31")
        .when(F.col("_flight_date") < F.col("_booking_date"), "flight date is before booking date")
        .when(F.col("_total_price").isNull() | (F.col("_total_price") <= 0), "total price must be greater than 0")
    )
)

valid_df = validated_df.where(F.col("validation_status").isNull()).drop("_key_count", "_recency_rank", "_row_id", "_parent_passengerid", "_booking_date", "_flight_date", "_total_price", "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop("_key_count", "_recency_rank", "_row_id", "_parent_passengerid", "_booking_date", "_flight_date", "_total_price").select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")

In [0]:
entity = "flight_booking"
key_cols = ["fb_bookingid", "fb_linenumber"]

bronze_tbl = f"{catalog}.bronze.{entity}"
silver_tbl = f"{catalog}.silver.{entity}"
quarantine_tbl = f"{catalog}.silver.{entity}_quarantine"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw_df = spark.table(bronze_tbl)

# Metadata columns to drop from the silver output
meta_cols = ["_source_file", "_ingested_at", "pipeline_run_id"]
keep_cols = [c for c in raw_df.columns if c not in meta_cols]

# Reference data from the silver layer
booking_ref = spark.table(f"{catalog}.silver.booking").select(
    F.col("bk_bookingid").alias("_bk_id"),
    F.to_date("bk_bookingdate", "yyyy-MM-dd").alias("_booking_date"),
)
route_ref = spark.table(f"{catalog}.silver.route").select(F.col("ro_routeid").alias("_rt_id"))
airline_ref = spark.table(f"{catalog}.silver.airline").select(F.col("al_airlineid").alias("_al_id"))
airline_route_ref = spark.table(f"{catalog}.silver.airline_route").select(
    F.col("ar_airlineid").alias("_ar_al_id"),
    F.col("ar_routeid").alias("_ar_rt_id"),
    F.col("ar_operatingcost").cast("double").alias("_operating_cost"),
)

# Validation rules:
# 1. booking id + line number composite key: no nulls; if duplicated, keep only the latest record
# 2. booking id, route id, airline id must exist in silver booking / route / airline
# 3. departure date must be between 1994-01-01 and 1998-12-31
# 4. departure date must be on or after the booking date (from silver booking)
# 5. arrival date must be on or after the departure date
# 6. ticket price must be greater than 0
# 7. discount must lie between 0 and 1
# 8. seats must be more than 0
# 9. operating cost (from silver airline_route) must be more than 0
key_count = Window.partitionBy(key_cols)
latest_rank = Window.partitionBy(key_cols).orderBy(F.col("_ingested_at").desc(), F.col("_row_id").desc())

raw_df = raw_df.withColumn("_row_id", F.monotonically_increasing_id())

validated_df = (
    raw_df
    .join(booking_ref, F.col("fb_bookingid") == F.col("_bk_id"), "left")
    .join(route_ref, F.col("fb_routeid") == F.col("_rt_id"), "left")
    .join(airline_ref, F.col("fb_airlineid") == F.col("_al_id"), "left")
    .join(airline_route_ref, (F.col("fb_airlineid") == F.col("_ar_al_id")) & (F.col("fb_routeid") == F.col("_ar_rt_id")), "left")
    .withColumn("_key_count", F.count(F.lit(1)).over(key_count))
    .withColumn("_recency_rank", F.row_number().over(latest_rank))
    .withColumn("_departure_date", F.to_date("fb_departuredate", "yyyy-MM-dd"))
    .withColumn("_arrival_date", F.to_date("fb_arrivaldate", "yyyy-MM-dd"))
    .withColumn("_ticket_price", F.col("fb_ticketprice").cast("double"))
    .withColumn("_discount", F.col("fb_discount").cast("double"))
    .withColumn("_seats", F.col("fb_seats").cast("double"))
    .withColumn(
        "validation_status",
        F.when(F.col("fb_bookingid").isNull() | F.col("fb_linenumber").isNull(), "booking id or line number is null")
        .when((F.col("_key_count") > 1) & (F.col("_recency_rank") > 1), "composite key duplicated - older record, latest kept")
        .when(F.col("_bk_id").isNull(), "booking id not found in silver booking")
        .when(F.col("_rt_id").isNull(), "route id not found in silver route")
        .when(F.col("_al_id").isNull(), "airline id not found in silver airline")
        .when(F.col("_departure_date").isNull(), "departure date is invalid or null")
        .when((F.col("_departure_date") < F.lit("1994-01-01")) | (F.col("_departure_date") > F.lit("1998-12-31")), "departure date is outside range 1994-01-01 to 1998-12-31")
        .when(F.col("_booking_date").isNull(), "booking date is invalid or null")
        .when(F.col("_departure_date") < F.col("_booking_date"), "departure date is before booking date")
        .when(F.col("_arrival_date").isNull(), "arrival date is invalid or null")
        .when(F.col("_arrival_date") < F.col("_departure_date"), "arrival date is before departure date")
        .when(F.col("_ticket_price").isNull() | (F.col("_ticket_price") <= 0), "ticket price must be greater than 0")
        .when(F.col("_discount").isNull() | (F.col("_discount") < 0) | (F.col("_discount") > 1), "discount must lie between 0 and 1")
        .when(F.col("_seats").isNull() | (F.col("_seats") <= 0), "seats must be more than 0")
       
    )
)

helper_cols = ["_key_count", "_recency_rank", "_row_id", "_bk_id", "_rt_id", "_al_id", "_ar_al_id", "_ar_rt_id", "_booking_date", "_departure_date", "_arrival_date", "_ticket_price", "_discount", "_seats", "_operating_cost"]

valid_df = validated_df.where(F.col("validation_status").isNull()).drop(*helper_cols, "validation_status").select(*keep_cols)
invalid_df = validated_df.where(F.col("validation_status").isNotNull()).drop(*helper_cols).select(*keep_cols, "validation_status")

valid_df.createOrReplaceTempView(f"{entity}_valid")
invalid_df.createOrReplaceTempView(f"{entity}_invalid")

spark.sql(f"""
CREATE OR REPLACE TABLE {silver_tbl} AS
SELECT * FROM {entity}_valid
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {quarantine_tbl} AS
SELECT * FROM {entity}_invalid
""")

print(f"Silver table:     {silver_tbl} ({spark.table(silver_tbl).count()} rows)")
print(f"Quarantine table: {quarantine_tbl} ({spark.table(quarantine_tbl).count()} rows)")